# PySpark Filtering + String/Date Operations + Null Handling

Topics:
- filter
- where
- String functions
- Date functions
- Null handling
- regexp_replace
- substring
- split
- concat

## 1. Imports

In [0]:
from datetime import date

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    DateType
)

## 2. Create Sample Orders Data

In [0]:
orders_data = [
    ("O101", "C101", "Rahul Sharma", "rahul.sharma@gmail.com",
     "Laptop", "Electronics", 1, 50000.0, date(2026, 9, 1)),

    ("O102", "C102", "Priya Kumar", "priya.kumar@yahoo.com",
     "Phone", "Electronics", 2, 30000.0, date(2026, 9, 2)),

    ("O103", "C101", "Rahul Sharma", "rahul.sharma@gmail.com",
     "Mouse", "Accessories", 3, 2000.0, date(2026, 9, 3)),

    ("O104", "C103", "Arjun Reddy", None,
     "Keyboard", "Accessories", 1, 3000.0, date(2026, 9, 3)),

    ("O105", "C102", "Priya Kumar", "priya.kumar@yahoo.com",
     "Monitor", "Electronics", 1, 15000.0, date(2026, 9, 4)),

    ("O106", "C101", "Rahul Sharma", "rahul.sharma@gmail.com",
     "Headphones", "Accessories", 2, 5000.0, date(2026, 9, 5)),

    ("O107", "C104", "Neha Singh", "neha.singh@company.com",
     "Tablet", "Electronics", 1, 25000.0, date(2026, 9, 5)),

    ("O108", "C105", None, "kiran123@gmail.com",
     "Phone", "Electronics", 1, 28000.0, date(2026, 9, 6)),

    ("O109", "C102", "Priya Kumar", "priya.kumar@yahoo.com",
     "Mouse", "Accessories", 2, 1500.0, None),

    ("O110", "C999", "Unknown Customer", "unknown@gmail.com",
     "Laptop", "Electronics", 1, 55000.0, date(2026, 9, 8))
]

## 3. Define Explicit Schema

In [0]:
orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("product", StringType(), True),
    StructField("category", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("amount", DoubleType(), True),
    StructField("order_date", DateType(), True)
])

## 4. Create Orders DataFrame

In [0]:
orders_df = spark.createDataFrame(
    orders_data,
    schema=orders_schema
)

orders_df.show(truncate=False)

In [0]:
df_filter = orders_df.filter( (F.col("amount")>20000 )  | (F.col("quantity")>1) ).show()

## 5. filter() — Amount Greater Than 20,000

In [0]:
high_value_orders_df = orders_df.filter(
    F.col("amount") > 20000
)

high_value_orders_df.show(truncate=False)

## 6. filter() — Multiple Conditions with AND

In [0]:
electronics_high_value_df = orders_df.filter(
    (F.col("category") == "Electronics") &
    (F.col("amount") > 20000)
)

electronics_high_value_df.show(truncate=False)

## 7. filter() — OR Condition

In [0]:
orders_or_df = orders_df.filter(
    (F.col("category") == "Electronics") |
    (F.col("amount") > 50000)
)

orders_or_df.show(truncate=False)

## 8. filter() — NOT Condition

In [0]:
non_electronics_df = orders_df.filter(
    "quantity == 2"
)

non_electronics_df.show(truncate=False)

## 9. where() — DataFrame Expression

In [0]:
where() is effectively an alias for filter() in the DataFrame API.

df.where(condition)

and:

df.filter(condition)

are equivalent for DataFrame filtering.

Why have both?
Because Spark supports both programming styles:

DataFrame style
df.filter(F.col("amount") > 20000)

SQL-like style
df.where(F.col("amount") > 20000)

where() is especially intuitive for people transitioning from SQL.

Example
orders_df.where( F.col("amount") > 20000 ).show()

Same result as:

orders_df.filter( F.col("amount") > 20000 ).show() # This seen above

Using SQL expression You can also write:

orders_df.where( "amount > 20000").show()

For complex production code, many teams prefer explicit F.col() expressions because they are easier to compose programmatically.

In [0]:
orders_where_df = orders_df.where(
    F.col("amount") > 20000
)

orders_where_df.show(truncate=False)

## 10. where() — SQL-Like String Expression

In [0]:
orders_where_sql_df = orders_df.where(
    "amount > 20000"
)

orders_where_sql_df.show(truncate=False)

## Strings

In [0]:
String manipulation is extremely common in Data Engineering.

Real source data often contains:

" rahul sharma "
"RAHUL SHARMA"
"Rahul-Sharma"
"john.doe@gmail.com"
"IND-1001"
"Hyderabad,India"

Before loading data into a curated layer, we may need to clean or transform it.

Important functions include:

upper()
lower()
trim()
ltrim()
rtrim()
length()
concat()
concat_ws()
substring()
split()
regexp_replace()
regexp_extract()

We'll focus on the requested functions and a few essential supporting functions.

## 11. upper() — Convert Customer Name to Uppercase

In [0]:
orders_df.select(
    "order_id",
    "customer_name",
    F.upper(F.col("customer_name")).alias("customer_name_upper"),
    F.lower(F.col("customer_name")).alias("customer_name_lower"),
    F.trim(F.col("customer_name")).alias("customer_name_trim"),
    F.ltrim(F.col("customer_name")).alias("customer_name_lefttrim"),
    F.rtrim(F.col("customer_name")).alias("customer_name_righttrim")
).show(truncate=False)

## 12. lower() — Normalize Email

In [0]:
orders_df.select(
    "order_id",
    "email",
    F.lower(F.col("email")).alias("email_lower")
).show(truncate=False)

## 13. trim() — Remove Leading/Trailing Spaces

In [0]:
orders_df.select(
    "order_id",
    F.trim(F.col("customer_name")).alias("clean_name")
).show(truncate=False)

## 14. length() — String Length

In [0]:
orders_df.select(
    "order_id",
    "product",
    F.length(F.col("product")).alias("product_length")
).show()

In [0]:
product - __Laptop*_

substring("product",3,6)

## 15. substring() — Extract Part of Order ID

In [0]:
orders_df.select(
    "order_id",
    F.substring("order_id", 3,2).alias("order_prefix")
).show()

## 16. split() — Split Customer Name

In [0]:
orders_df.select(
    "order_id",
    "customer_name",
    F.split( F.col("customer_name"), " " ).alias("name_parts")
).show(truncate=False)

## 17. split() — Extract First Name

In [0]:
orders_df.select(
    "customer_name",
    F.split(
        F.col("customer_name"),
        " "
    )[0].alias("first_name")
).show()

## 18. split() — Extract Last Name

In [0]:
orders_df.select(
    "customer_name",
    F.split(
        F.col("customer_name"),
        " "
    )[1].alias("last_name")
).show()

In [0]:
orders_df.show()

## 19. concat() — Combine Customer ID and Product

In [0]:
orders_df.select(
    "customer_id",
    "product",
    F.concat(
        F.col("customer_id"),
        F.col("product"),F.col("customer_id"),F.col("product")
    ).alias("customer_product")
).show(truncate=False)

## 20. concat() + lit() — Add Separator

In [0]:
orders_df.select(
    "customer_id",
    "product",
    F.concat(
        F.col("customer_id"),
        F.lit("-"),
        F.col("product")
    ).alias("customer_product")
).show()

## 21. concat_ws() — Concatenate with Separator

In [0]:
orders_df.select(
    "customer_id",
    "product",
    F.concat_ws(
        "|",
        F.col("customer_id"),
        F.col("product"),F.col("customer_id")
    ).alias("customer_product")
).show()

### regexp_replace()

In [0]:
regexp_replace() : regexp_replace() replaces portions of a string using a regular expression.

This is extremely important in production Data Engineering.

WHY?

Real source data might contain:

+91-98765-43210
(91) 98765 43210
91 98765 43210

We may want:

919876543210

Regular expressions allow us to perform flexible pattern-based cleansing.

Syntax
F.regexp_replace(
    column,
    pattern,
    replacement
)

## 22. regexp_replace() — Clean Phone Numbers

In [0]:
# Create a small customer DataFrame:
phone_data = [
    ("C101", "+91-98765-43210"),
    ("C102", "91 98765 43211"),
    ("C103", "(91) 98765-43212"),
    ("C104", "9876543213")
]

phone_df = spark.createDataFrame(
    phone_data,
    ["customer_id", "phone"]
)

phone_df.show()

In [0]:
clean_phone_df = phone_df.withColumn(
    "clean_phone",F.substring( \
    F.regexp_replace(
        F.col("phone"),
        r"[^0-9]",
        ""
    ),3,10)
)

# clean_phone_df.show()

clean_phone_df.printSchema()

In [0]:
orders_df.show()

## 23. regexp_replace() — Clean Special Characters

In [0]:
orders_df.select(
    "customer_name",
    F.regexp_replace(
        F.col("customer_name"),
        r"[^a-zA-Z ]",
        ""
    ).alias("clean_name")
).show(truncate=False)

## 24. current_date() — Current Spark Date

In [0]:
orders_df.select(
    F.current_date().alias("today")
).show()

## 25. year() — Extract Year

In [0]:
orders_df.select(
    "order_id",
    "order_date",
    F.year("order_date").alias("order_year")
).show()

## 26. month() — Extract Month

In [0]:
orders_df.select(
    "order_id",
    "order_date",
    F.month("order_date").alias("order_month")
).show()

## 27. day() — Extract Day

In [0]:
orders_df.select(
    "order_id",
    "order_date",
    F.day("order_date").alias("order_day")
).show()

In [0]:
date_add(col_name,1)

## 28. date_add() — Add Days

In [0]:
orders_df.select(
    "order_id",
    "order_date",
    F.date_add(
        F.col("order_date"),
        7
    ).alias("delivery_target_date")
).show()

## 29. date_sub() — Subtract Days

In [0]:
orders_df.select(
    "order_id",
    "order_date",
    F.date_sub(
        F.col("order_date"),
        7
    ).alias("previous_week")
).show()

## 30. datediff() — Days Since Order

In [0]:
#Calculates difference between dates.
orders_df.select(
    "order_id",
    "order_date",
    F.datediff(
        F.current_date(),
        F.col("order_date")
    ).alias("days_since_order")
).show()

In [0]:
This datediff() is useful for:

SLA calculations
aging reports
order ageing
customer activity
retention analysis

 Service Level Agreement (SLA) - An SLA is a formal contract between a service provider and a customer.

## 31. date_format() — Create Year-Month

In [0]:
Suppose we want 2026-09 from 2026-09-01

In [0]:
orders_df.select(
    "order_id",
    "order_date",
    F.date_format(
        F.col("order_date"),
        "yyyy-MM"
    ).alias("order_month")
).show()

In [0]:
NULL - ijasdchsbd

NULL - sdcrfvascb

""

" "

### NULLS

In [0]:
Null Handling is one of the most important Data Engineering topics.

Our dataset intentionally contains:

O104 → email = NULL
O108 → customer_name = NULL
O109 → order_date = NULL

NULL means: The value is missing/unknown/not available.

It is not the same as: 

""

and not necessarily:

0

and not necessarily:

"NULL"

These are different concepts.

## 32. isNull() — Find NULL Emails

In [0]:
orders_df.filter(
    F.col("email").isNull() 
).show(truncate=False)

## 33. isNotNull() — Find Non-NULL Emails

In [0]:
orders_df.filter(
    F.col("email").isNotNull()
).show(truncate=False)

In [0]:
orders_df.filter(
    F.col("email").isNotNull()
).show()

## 34. fillna() — Replace NULL Values

In [0]:
orders_filled_df = orders_df.fillna({
    "customer_name": "Unknown",
    "email": "Not Available"
})

orders_filled_df.select(
    "order_id",
    "customer_name",
    "email"
).show(truncate=False)

In [0]:
O107    | NULL   , NULL  |neha.singh@company.com |  -  fillna() - Unknown , Unknown 

O107    |Neha   , NULL    |neha.singh@company.com - coalesce() - Neha , Unknown Customer 


Country - INDIA 

1m RECORDS - 100k - Nulls - country


F.col(country).fillna("India")



In [0]:
from pyspark.sql import functions as F

# Sample data with missing emails
data = [
    ("Alice", "alice@gmail.com", None),
    ("Bob", None, "bob_backup@gmail.com"),
    ("Charlie", None, None)
]
df1 = spark.createDataFrame(data, ["name", "primary_email", "secondary_email"]).show()


## 35. coalesce() — Returns First Non-NULL Value

In [0]:
orders_coalesced_df = orders_df.select(
    "order_id",
    F.coalesce(
        F.col("customer_name"),
        F.lit("Unknown Customer")
    ).alias("customer_name")
)

orders_coalesced_df.show(truncate=False)

## 36. when() — Conditional NULL Handling

In [0]:
orders_clean_name_df = orders_df.withColumn(
    "customer_name_clean",
    F.when(
        F.col("customer_name").isNull(),
        "Unknown Customer"
    ).otherwise(
        F.col("customer_name")
    )
)

orders_clean_name_df.select(
    "order_id",
    "customer_name_clean"
).show(truncate=False)

In [0]:
orders_df.show(truncate=False)

## 37. dropna() — Remove Rows with NULL Email

In [0]:
orders_no_null_email_df = orders_df.dropna(
    subset=["email"]
)

orders_no_null_email_df.show(truncate=False)

In [0]:
Be careful with dropna()
In production, don't blindly do: df.dropna() because you may unintentionally remove a huge number of legitimate records.

Instead use:

df.dropna(
    subset=["customer_id"]
)

if customer_id is a mandatory business key.

## 38. Real-World Combined Transformation

In [0]:
production_filter_df = (
    orders_df
    .filter(
        (F.col("category") == "Electronics") &
        (F.col("amount") > 20000)
    )
    .withColumn(
        "customer_name_clean",
        F.coalesce(
            F.trim(F.col("customer_name")),
            F.lit("Unknown Customer")
        )
    )
    .withColumn(
        "email_clean",
        F.lower(F.trim(F.col("email")))
    )
    .withColumn(
        "order_month",
        F.date_format(
            F.col("order_date"),
            "yyyy-MM"
        )
    )
)

production_filter_df.select(
    "order_id",
    "customer_name_clean",
    "email_clean",
    "product",
    "amount",
    "order_month"
).show(truncate=False)

## 39. Inspect the Logical and Physical Plan

Use `explain(True)` to connect the DataFrame transformations to Spark's internal planning.

Look for:
- Parsed Logical Plan
- Analyzed Logical Plan
- Optimized Logical Plan
- Physical Plan

Remember: DataFrame transformations are lazy; an action such as `show()` triggers execution.

In [0]:
production_filter_df.explain(True)

## Key Production Takeaways

1. `filter()` and `where()` are equivalent DataFrame filtering APIs.
2. Use `&`, `|`, and `~` for Spark Column boolean expressions.
3. Use `isNull()` / `isNotNull()` for NULL checks.
4. `NULL` is different from an empty string or zero.
5. Use Spark string functions instead of Python loops for large datasets.
6. Use `regexp_replace()` for pattern-based cleansing.
7. `split()` returns an array.
8. `substring()` uses Spark SQL's 1-based position semantics.
9. Date functions are essential for partitioning, reporting, SLA and aging logic.
10. Be deliberate with `fillna()` and `dropna()` so valid records are not accidentally removed.
11. Filtering early can reduce the amount of data carried into later transformations, and compatible predicates may be pushed toward supported data sources.
12. Always validate complex transformations using `show()`, `printSchema()` and `explain(True)`.

In [0]:
| Concept            | Definition                        | Syntax                    |
| ------------------ | --------------------------------- | ------------------------- |
| `filter()`         | Keeps rows satisfying a condition | `df.filter(condition)`    |
| `where()`          | Alias for DataFrame filtering     | `df.where(condition)`     |
| `upper()`          | Converts text to uppercase        | `F.upper("name")`         |
| `lower()`          | Converts text to lowercase        | `F.lower("email")`        |
| `trim()`           | Removes leading/trailing spaces   | `F.trim("name")`          |
| `substring()`      | Extracts part of a string         | `F.substring("id",1,3)`   |
| `split()`          | Splits string into an array       | `F.split("name"," ")`     |
| `concat()`         | Combines strings                  | `F.concat(...)`           |
| `concat_ws()`      | Combines strings with separator   | `F.concat_ws("-",...)`    |
| `regexp_replace()` | Replaces regex matches            | `F.regexp_replace(...)`   |
| `current_date()`   | Current date                      | `F.current_date()`        |
| `year()`           | Extract year                      | `F.year("order_date")`    |
| `month()`          | Extract month                     | `F.month("order_date")`   |
| `day()`            | Extract day                       | `F.day("order_date")`     |
| `date_add()`       | Add days                          | `F.date_add("date",7)`    |
| `date_sub()`       | Subtract days                     | `F.date_sub("date",7)`    |
| `datediff()`       | Difference between dates          | `F.datediff(date1,date2)` |
| `date_format()`    | Format date                       | `F.date_format(...)`      |
| `isNull()`         | Check NULL                        | `F.col("x").isNull()`     |
| `isNotNull()`      | Check non-NULL                    | `F.col("x").isNotNull()`  |
| `fillna()`         | Replace NULL values               | `df.fillna(...)`          |
| `dropna()`         | Remove NULL-containing rows       | `df.dropna(...)`          |
| `coalesce()`       | First non-null value              | `F.coalesce(...)`         |


In [0]:
                 RAW DATA
                    |
                    v
              DataFrame
                    |
                    v
          FILTER UNNECESSARY ROWS
                    |
                    v
           CLEAN STRING DATA
          /        |         \
       trim      regexp      split
          \        |         /
           \       |        /
              CLEAN DATA
                    |
                    v
             HANDLE NULLS
                    |
                    v
             DATE PROCESSING
          /        |         \
       year      month      datediff
                    |
                    v
             CURATED DATA
                    |
                    v
              SILVER LAYER

coalesce use case

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import coalesce, lit

#  Let's create our friends' profiles (None means it's totally empty!)
player_data = [
    ("Player_1", "X_Ninja_X", "Alex", "User#101"),  # Has a Gamertag!
    ("Player_2", None, "Emma", "User#202"),       # No Gamertag, but has a First Name!
    ("Player_3", None, None, "User#303"),          # Secret player, only has an Account Number!
    ("Player_4", None, None, None)                 # Brand new player who filled out nothing!
]

columns = ["Player", "Gamertag", "FirstName", "AccountNumber"]
df = spark.createDataFrame(player_data, columns)
df.show()



In [0]:
df.show()
#  Use coalesce() to find what to display on the leaderboard!
# lit("Guest") is our ultimate backup plan if everything else is empty.
scoreboard_df = df.withColumn(
    "Displayed_Name", 
    coalesce(df["Gamertag"], df["FirstName"], df["AccountNumber"], lit("Guest"))
)

# Show the final scoreboard!
scoreboard_df.select("Player", "Displayed_Name").show()

In [0]:
+--------+--------------+

|  Player|Displayed_Name|
+--------+--------------+

|Player_1|     X_Ninja_X|  <-- Grabs Gamertag because it exists
|Player_2|          Emma|  <-- Gamertag was empty, grabs First Name
|Player_3|       User#303|  <-- Gamertag & Name were empty, grabs Account #
|Player_4|         Guest|  <-- Everything was empty, falls back to "Guest"!
+--------+--------------+
